# Website Video Generator — Google Colab MVP

Masukkan URL website publik, pilih template, lalu buat preview dan video MP4.

**Catatan:** GPU opsional. HyperFrames juga opsional; notebook memiliki renderer FFmpeg fallback agar MVP tetap dapat berjalan.

In [ ]:
import os, sys, subprocess, shutil, platform, json, re, time, math, tempfile, importlib.util
from pathlib import Path

PROJECT = Path('/content/website_video_generator')
for name in ['captures','output','temp','compositions']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)

REQUIRED_PACKAGES = [
    ('gradio>=4.0', 'gradio'),
    ('playwright', 'playwright'),
    ('requests', 'requests'),
    ('beautifulsoup4', 'bs4'),
    ('Pillow', 'PIL'),
    ('imageio-ffmpeg', 'imageio_ffmpeg'),
    ('psutil', 'psutil'),
]

missing_packages = [pkg for pkg, module_name in REQUIRED_PACKAGES if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print(f'[setup] Installing missing Python packages: {missing_packages}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('[setup] Python packages already available; skip pip install.')

APT_PACKAGES = [
    'libatk1.0-0','libatk-bridge2.0-0','libcups2','libdrm2','libxkbcommon0',
    'libxcomposite1','libxdamage1','libxfixes3','libxrandr2','libgbm1',
    'libasound2','libnss3','libnspr4','libgtk-3-0'
]

def run_cmd(cmd, check=False):
    print('[setup] $', ' '.join(cmd))
    return subprocess.run(cmd, text=True)

print('[setup] Installing Playwright Chromium (preferred: --with-deps)...')
chromium_ready = run_cmd([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium']).returncode == 0

if not chromium_ready:
    print('[setup][warn] playwright install --with-deps chromium gagal; mencoba fallback yang aman...')
    if shutil.which('apt-get'):
        run_cmd(['apt-get', 'update'])
        apt_result = run_cmd(['apt-get', 'install', '-y', *APT_PACKAGES])
        if apt_result.returncode != 0:
            print('[setup][warn] apt install dependency Chromium gagal. Notebook tetap lanjut, cek troubleshooting jika Generate gagal.')
    else:
        print('[setup][warn] apt-get tidak tersedia pada runtime ini.')
    chromium_ready = run_cmd([sys.executable, '-m', 'playwright', 'install', 'chromium']).returncode == 0

print('[setup] Chromium install status:', 'OK' if chromium_ready else 'BELUM SIAP')
print('Project:', PROJECT)
print('FFmpeg:', shutil.which('ffmpeg') or 'imageio-ffmpeg fallback')
print('Optional HyperFrames install can be attempted separately; it is not required.')


In [ ]:
import psutil
print('Python:', sys.version)
print('Platform:', platform.platform())
print('RAM GB:', round(psutil.virtual_memory().total/1024**3, 2))
print('Disk free GB:', round(shutil.disk_usage('/content').free/1024**3, 2))
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'Tidak tersedia / CPU runtime')

In [ ]:
import asyncio, threading, requests, html
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from PIL import Image, ImageDraw, ImageFont

UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36'
BROWSER_STATUS = {'ok': False, 'checked': False}


def normalize_url(value):
    value = (value or '').strip()
    if not re.match(r'^https?://', value, re.I): value = 'https://' + value
    parsed = urlparse(value)
    if not parsed.netloc: raise ValueError('URL tidak valid.')
    return value


def validate_url(value):
    url = normalize_url(value)
    response = requests.get(url, headers={'User-Agent': UA}, timeout=20, allow_redirects=True)
    if response.status_code >= 400: raise ValueError(f'HTTP {response.status_code}: website tidak dapat diakses.')
    return response.url


def chromium_executable_path():
    candidates = [
        os.environ.get('PLAYWRIGHT_CHROMIUM_EXECUTABLE_PATH', ''),
        '/usr/bin/chromium-browser',
        '/usr/bin/chromium',
        '/usr/bin/google-chrome',
        '/opt/google/chrome/chrome',
    ]
    for path in candidates:
        if path and Path(path).exists():
            return path
    return None


def has_gpu_runtime():
    if not shutil.which('nvidia-smi'):
        return False
    result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
    return result.returncode == 0 and bool(result.stdout.strip())


def chromium_launch_args():
    args = ['--no-sandbox', '--disable-dev-shm-usage']
    if not has_gpu_runtime():
        args += ['--disable-gpu', '--disable-software-rasterizer']
    return args


def chromium_launch_kwargs():
    kwargs = {'headless': True, 'args': chromium_launch_args()}
    executable = chromium_executable_path()
    if executable:
        kwargs['executable_path'] = executable
    return kwargs


def browser_dependency_help(raw_error):
    if any(token in raw_error for token in ['libatk-1.0.so.0', 'BrowserType.launch', 'cannot open shared object file']):
        return (
            'Chromium gagal start karena dependency sistem belum lengkap.\n'
            'Coba jalankan ulang cell instalasi. Jika masih gagal, jalankan manual:\n'
            '!apt-get update && apt-get install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libnss3 libnspr4 libgtk-3-0\n'
            '!python -m playwright install chromium'
        )
    return 'Pastikan cell instalasi selesai tanpa error dan Chromium terpasang.'


async def browser_preflight_async():
    launch_kwargs = chromium_launch_kwargs()
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(**launch_kwargs)
            page = await browser.new_page(viewport={'width': 1280, 'height': 720}, user_agent=UA)
            await page.goto('https://example.com', wait_until='domcontentloaded', timeout=30000)
            await browser.close()
        return {
            'ok': True,
            'checked': True,
            'executable': launch_kwargs.get('executable_path') or 'bundled-playwright-chromium',
            'args': launch_kwargs['args'],
        }
    except Exception as exc:
        raw_error = str(exc)
        return {
            'ok': False,
            'checked': True,
            'error': raw_error,
            'hint': browser_dependency_help(raw_error),
            'executable': launch_kwargs.get('executable_path') or 'bundled-playwright-chromium',
            'args': launch_kwargs['args'],
        }


def run_coro_sync(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    holder = {}

    def runner():
        try:
            holder['value'] = asyncio.run(coro)
        except Exception as exc:
            holder['error'] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if 'error' in holder:
        raise holder['error']
    return holder['value']


def ensure_browser_ready(force=False):
    global BROWSER_STATUS
    if BROWSER_STATUS.get('checked') and BROWSER_STATUS.get('ok') and not force:
        return BROWSER_STATUS
    print('[browser-check] Menjalankan preflight Chromium...')
    BROWSER_STATUS = run_coro_sync(browser_preflight_async())
    if BROWSER_STATUS['ok']:
        print('[browser-check] OK | executable:', BROWSER_STATUS['executable'])
    else:
        print('[browser-check][error]', BROWSER_STATUS.get('error', 'Unknown error'))
        print('[browser-check][hint]', BROWSER_STATUS.get('hint', '-'))
    return BROWSER_STATUS


async def capture(url, folder, width, height, steps=5):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(**chromium_launch_kwargs())
            page = await browser.new_page(viewport={'width': width, 'height': height}, user_agent=UA)
            await page.goto(url, wait_until='domcontentloaded', timeout=60000)
            await page.wait_for_timeout(2500)
            page_height = await page.evaluate('Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)')
            max_scroll = max(0, page_height-height)
            positions = [0] if max_scroll == 0 else [round(max_scroll*i/max(steps-1,1)) for i in range(steps)]
            shots = []
            for i, pos in enumerate(positions):
                await page.evaluate('(y) => window.scrollTo(0,y)', pos)
                await page.wait_for_timeout(700)
                path = folder/f'capture_{i:02d}.png'
                await page.screenshot(path=str(path), full_page=False)
                shots.append({'path': str(path), 'scroll_y': pos, 'index': i})
            await browser.close()
        return {'url': url, 'page_height': page_height, 'screenshots': shots}
    except Exception as exc:
        raw_error = str(exc)
        raise RuntimeError(f'{raw_error}\n\n{browser_dependency_help(raw_error)}') from exc


In [ ]:
def analyze(url):
    try:
        text = requests.get(url, headers={'User-Agent': UA}, timeout=20).text
        soup = BeautifulSoup(text, 'lxml')
        headings = [x.get_text(' ', strip=True) for x in soup.find_all(['h1','h2','h3'])]
        blob = ' '.join(headings).lower()
        labels = []
        for key, words in {'hero':['hero','welcome','build','business'], 'features':['feature','solution','everything'], 'product':['product','platform','dashboard'], 'testimonial':['testimonial','customer','stories'], 'cta':['start','trial','contact','learn more']}.items():
            if any(w in blob for w in words): labels.append(key)
        return {'headings': headings[:12], 'sections': labels}
    except Exception as e:
        return {'headings': [], 'sections': [], 'warning': str(e)}

def storyboard(meta, duration, mode, title, brand, cta):
    shots = meta['screenshots']
    opening = min(3, duration*0.12); closing = min(3, duration*0.12)
    middle = max(1, duration-opening-closing) / max(1, len(shots))
    hero_title = (title or '').strip() or (brand or '').strip() or 'Website Showcase'
    scenes = [{'name':'opening','duration':opening,'image':None,'text':hero_title}]
    for i, shot in enumerate(shots):
        label = 'hero' if i == 0 else ('cta' if i == len(shots)-1 else f'section_{i}')
        text = cta if label == 'cta' else (hero_title if label == 'hero' else '')
        scenes.append({'name':label,'duration':middle,'image':shot['path'],'text':text})
    scenes.append({'name':'closing','duration':closing,'image':None,'text':cta or brand or 'Learn more'})
    return {'mode': mode, 'scenes': scenes, 'total': sum(x['duration'] for x in scenes)}



In [ ]:
def _validate_size(width, height):
    width, height = int(width), int(height)
    if width <= 0 or height <= 0:
        raise ValueError(f'Ukuran frame tidak valid: {width}x{height}')
    return width, height


def _safe_rgb_color(bg):
    value = (bg or '#0f172a').strip()
    if not value:
        return '#0f172a'
    return value


def make_frame(scene, size, bg, brand, zoom=1.0, mockup=True):
    w, h = _validate_size(*size)
    canvas = Image.new('RGBA', (w, h), _safe_rgb_color(bg))
    draw = ImageDraw.Draw(canvas)
    if scene.get('image'):
        src = Path(scene['image'])
        if not src.exists() or src.stat().st_size == 0:
            raise ValueError(f'File screenshot tidak valid: {src}')
        with Image.open(src) as raw:
            raw.load()
            img = raw.convert('RGBA')
        target_w = int(w*.84) if mockup else int(w*.96)
        target_h = int(h*.62)
        img.thumbnail((max(8, target_w), max(8, target_h)))
        x = max(0, (w-img.width)//2)
        y = max(0, int(h*.16))
        if mockup:
            draw.rounded_rectangle((x-8, y-38, x+img.width+8, y+img.height+8), radius=16, fill='#1f2937')
            for dot, color in enumerate(['#ef4444', '#f59e0b', '#22c55e']):
                draw.ellipse((x+10+dot*18, y-27, x+20+dot*18, y-17), fill=color)
            y += 4
        canvas.alpha_composite(img, (x, y))
    if brand:
        draw.text((int(w*.07), int(h*.06)), str(brand)[:80], fill='white')
    text = str(scene.get('text','') or '')
    if text:
        draw.text((int(w*.08), int(h*.84)), text[:90], fill='white')
    return canvas.convert('RGB')


def _validate_png(path, expected_size):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f'Frame PNG kosong/tidak ada: {path}')
    with Image.open(path) as probe:
        probe.verify()
    with Image.open(path) as probe:
        if probe.size != expected_size:
            raise ValueError(f'Ukuran frame tidak konsisten: {probe.size} != {expected_size}')
        mode = probe.mode
    if mode not in ('RGB', 'RGBA'):
        raise ValueError(f'Mode frame tidak didukung: {mode}')


def _tail_text(value, max_chars=3000):
    text = (value or '').strip()
    if len(text) <= max_chars:
        return text
    return '...
' + text[-max_chars:]


def _run_ffmpeg_encode(ffmpeg, concat_path, output_path, fps, codec='libx264'):
    output_path = Path(output_path)
    base = [
        str(ffmpeg), '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_path),
        '-vf', f'fps={int(fps)},format=yuv420p', '-movflags', '+faststart'
    ]
    if codec == 'libx264':
        cmd = base + ['-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23', str(output_path)]
    else:
        cmd = base + ['-c:v', 'mpeg4', '-q:v', '4', str(output_path)]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return {'ok': proc.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0, 'cmd': cmd, 'stderr': proc.stderr, 'stdout': proc.stdout, 'code': proc.returncode}


def render_video(sb, output, width, height, fps=24, bg='#0f172a', brand='', mockup=True):
    width, height = _validate_size(width, height)
    fps = max(12, min(30, int(fps)))
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)

    ffmpeg = shutil.which('ffmpeg')
    if not ffmpeg:
        import imageio_ffmpeg
        ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

    job_dir = Path(tempfile.mkdtemp(prefix='render_job_', dir=str(PROJECT/'temp')))
    frame_dir = job_dir / 'frames'
    frame_dir.mkdir(parents=True, exist_ok=True)
    concat_file = job_dir / 'frames.txt'

    try:
        concat_lines = []
        created = 0
        for i, scene in enumerate(sb['scenes']):
            frame_path = frame_dir / f'scene_{i:03d}.png'
            frame = make_frame(scene, (width, height), bg, brand, mockup=mockup)
            frame.save(frame_path, format='PNG', optimize=True)
            _validate_png(frame_path, (width, height))
            duration = max(0.12, float(scene.get('duration', 0.12)))
            concat_lines.append(f"file '{frame_path.as_posix()}'")
            concat_lines.append(f'duration {duration:.4f}')
            created += 1
        if created == 0:
            raise ValueError('Storyboard tidak menghasilkan frame.')

        last_frame = frame_dir / f'scene_{created-1:03d}.png'
        concat_lines.append(f"file '{last_frame.as_posix()}'")
        concat_file.write_text('
'.join(concat_lines) + '
', encoding='utf-8')

        attempts = []
        for codec in ['libx264', 'mpeg4']:
            result = _run_ffmpeg_encode(ffmpeg, concat_file, output, fps, codec=codec)
            attempts.append(result)
            if result['ok']:
                return str(output)

        detail = []
        for idx, attempt in enumerate(attempts, 1):
            detail.append(
                f"Attempt {idx} | returncode={attempt['code']}\n"
                f"Command: {' '.join(attempt['cmd'])}\n"
                f"stderr (tail):\n{_tail_text(attempt['stderr'])}"
            )
        raise RuntimeError('FFmpeg encode gagal pada semua fallback.\n\n' + '\n\n'.join(detail))
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)



In [ ]:
print('Mengimpor Gradio (ini hanya import package, bukan render UI)...')
import gradio as gr
print('Import Gradio selesai.')


def _base_size(orientation, resolution):
    if orientation == 'Landscape 16:9':
        base = (1920, 1080)
    else:
        base = (1080, 1920)
    if resolution == '720p':
        return (1280, 720) if orientation.startswith('Landscape') else (720, 1280)
    return base


def _preview_size(base):
    w, h = base
    max_edge = max(w, h)
    if max_edge <= 960:
        return (w, h)
    scale = 960.0 / max_edge
    return (max(256, int(w*scale)), max(256, int(h*scale)))


def _render_fps(speed, resolution, orientation):
    raw = {'Slow': 18, 'Medium': 24, 'Fast': 30}.get(speed, 24)
    if resolution == '1080p' and orientation.startswith('Portrait'):
        raw = min(raw, 24)
    return max(12, min(30, int(raw)))


def _save_preview_png(scene, output_path, size, bg, brand, mockup):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame = make_frame(scene, size, bg, brand, mockup=mockup)
    frame.save(output_path, format='PNG', optimize=True)
    _validate_png(output_path, tuple(size))
    return str(output_path)


def generate(url, mode, orientation, duration, resolution, title, brand, cta, bg, speed, mockup, zoom, action):
    progress = []

    def mark(stage, message):
        progress.append({'stage': stage, 'message': message, 'ts': round(time.time(), 3)})

    try:
        mark('validate', 'Memeriksa browser runtime')
        browser_status = ensure_browser_ready()
        if not browser_status.get('ok'):
            raise RuntimeError(browser_status.get('hint', 'Browser Chromium belum siap.'))

        mark('validate', 'Validasi URL dan redirect')
        final_url = validate_url(url)
        base = _base_size(orientation, resolution)
        preview_size = _preview_size(base)
        fps = _render_fps(speed, resolution, orientation)

        capture_dir = PROJECT / 'captures' / f"{int(time.time()*1000)}_{os.getpid()}"
        mark('capture', f'Capture website ke {capture_dir.name}')
        meta = run_coro_sync(capture(final_url, capture_dir, 1280, 720, 5))

        mark('storyboard', 'Menganalisis halaman dan menyusun storyboard')
        analysis = analyze(final_url)
        sb = storyboard(meta, int(duration), mode, title, brand, cta)
        if not analysis.get('sections'):
            mark('storyboard', 'analysis.sections kosong; lanjut pakai hasil capture')

        preview_scene = sb['scenes'][1] if len(sb['scenes']) > 1 else sb['scenes'][0]
        preview_path = PROJECT / 'output' / f'preview_{int(time.time()*1000)}.png'

        mark('preview', f'Generate preview PNG {preview_size[0]}x{preview_size[1]}')
        preview = _save_preview_png(preview_scene, preview_path, preview_size, bg, brand, mockup)

        payload = {
            'status': {'current': 'preview', 'events': progress},
            'url': final_url,
            'analysis': analysis,
            'storyboard': sb,
            'render': {'target_size': {'width': base[0], 'height': base[1]}, 'fps': fps},
            'artifacts': {'preview_png': preview, 'video_mp4': None}
        }

        if action == 'Generate Preview':
            payload['status']['current'] = 'done_preview'
            return preview, None, json.dumps(payload, indent=2, ensure_ascii=False)

        mark('render', 'Rendering MP4 dengan fallback codec bila perlu')
        out = PROJECT / 'output' / f'website_showcase_{int(time.time()*1000)}.mp4'
        video = render_video(sb, out, *base, fps=fps, bg=bg, brand=brand, mockup=mockup)
        if not Path(video).exists() or Path(video).stat().st_size == 0:
            raise RuntimeError(f'Output video tidak ditemukan/kosong: {video}')

        payload['status']['current'] = 'done_render'
        payload['status']['events'] = progress
        payload['artifacts']['video_mp4'] = video
        return preview, video, json.dumps(payload, indent=2, ensure_ascii=False)
    except Exception as e:
        error_payload = {
            'status': {'current': 'error', 'events': progress},
            'error': str(e)
        }
        return None, None, json.dumps(error_payload, indent=2, ensure_ascii=False)

with gr.Blocks(title='Website Video Generator') as demo:
    gr.Markdown('# Website Video Generator')
    gr.Markdown('ℹ️ Startup awal dapat memakan waktu karena preflight browser. Gunakan `share=True` hanya saat butuh link publik.')
    with gr.Row():
        url=gr.Textbox(label='URL website', placeholder='https://example.com')
        mode=gr.Dropdown(['Website Tour','Cinematic Showcase','Hybrid Showcase'], value='Hybrid Showcase', label='Mode')
    with gr.Row():
        orientation=gr.Dropdown(['Landscape 16:9','Portrait 9:16'], value='Portrait 9:16', label='Orientasi')
        duration=gr.Dropdown([15,30,45,60], value=30, label='Durasi (detik)')
        resolution=gr.Dropdown(['720p','1080p'], value='720p', label='Resolusi')
    with gr.Row():
        title=gr.Textbox(label='Judul'); brand=gr.Textbox(label='Brand'); cta=gr.Textbox(label='CTA')
    with gr.Row():
        bg=gr.ColorPicker(value='#0f172a', label='Background'); speed=gr.Dropdown(['Slow','Medium','Fast'], value='Medium', label='Kecepatan scroll'); mockup=gr.Checkbox(value=True,label='Browser mockup'); zoom=gr.Checkbox(value=True,label='Efek zoom')
    action=gr.Radio(['Generate Preview','Render Video'], value='Generate Preview', label='Aksi')
    run=gr.Button('Jalankan', variant='primary')
    with gr.Row():
        preview=gr.Image(label='Preview', type='filepath')
        video=gr.Video(label='Video MP4')
    details=gr.Code(label='Detail storyboard / status', language='json')
    run.click(generate, [url,mode,orientation,duration,resolution,title,brand,cta,bg,speed,mockup,zoom,action], [preview,video,details])

_browser_status = ensure_browser_ready()
if _browser_status.get('ok'):
    print('Gradio sedang memulai server lokal (share=False). Ubah ke share=True jika butuh link publik.')
    demo.launch(share=False, show_error=True, quiet=False)
else:
    print('Gradio tidak dijalankan karena browser check gagal.')
    print(_browser_status.get('hint', _browser_status.get('error', 'Unknown browser error')))

